In [ ]:
import ROOT
import math

# print(f"ROOT version: {ROOT.__version__}")
import numpy as np
from scipy.special import j0
import plotly.graph_objects as go
from scipy.integrate import fixed_quad


In [ ]:
def read_data_file(filename):
    """Read data file and return arrays for x, y, y_error"""
    x_vals = []
    y_vals = []
    y_errs = []
    
    with open(filename, 'r') as f:
        for line in f:
            if line.strip() and not line.startswith('#'):
                parts = line.split()
                if len(parts) >= 3:
                    x_vals.append(float(parts[0]))
                    y_vals.append(float(parts[1]))
                    y_errs.append(float(parts[2]))
    
    return x_vals, y_vals, y_errs

In [ ]:
# Load experimental data for all energies from ATLAS
x_atlas_all, y_atlas_all, yerr_atlas_all = read_data_file('../../../data/ens_atlas_difc0_2.dat')

# Function to process data for each energy block
def process_data(x_data, y_data, yerr_data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        if end is None:
            end = len(x_data)
        x_values.append(x_data[start:end])
        y_values.append(y_data[start:end])
        y_errors.append(yerr_data[start:end])
    
    return x_values, y_values, y_errors


In [ ]:
#ranges for each energy 
atlas_blocks = [(0, 29), (29, 58), (58, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(x_atlas_all, y_atlas_all, yerr_atlas_all, atlas_blocks)

# Extract values by energy
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

In [ ]:
# defining parameters/constants
b_0 = (33 - 6) / (12 * np.pi)
lambda_qcd = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25


#ensemble parameters
param_mg_atlas_pl = 0.421
param_eps_atlas_pl = 0.0753
param_a1_atlas_pl = 1.517
param_a2_atlas_pl = 2.05

In [ ]:
#--------------------------------------
# Eq 22 - GE
#--------------------------------------
def m2_pl(q2, mg):
    lambda2 = lambda_qcd ** 2
    rho_mg_2 = rho * (mg ** 2)
    ratio = math.log((q2 + rho_mg_2) / lambda2) / math.log(rho_mg_2 / lambda2)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)

#--------------------------------------
# Eq 26 - GE
#--------------------------------------
def G_p(q2, a1, a2):
    t = -q2
    return np.exp(-(a1 * np.abs(t) + a2 * np.abs(t) ** 2))


#--------------------------------------
# Eq 24 - GE
#--------------------------------------
def alpha_D(q2, mg, m2_type):
    m2_func = m2_type(q2, mg)
    return 1.0 / (b_0 * (q2 + m2_func) * math.log((q2 + 4 * m2_func) / (lambda_qcd ** 2)))

#--------------------------------------
# Eq 7 - GE
#--------------------------------------
def T_1(k, phi, mg, a1, a2, m2_type, q):
    q2 = q ** 2
    qk_cos = q * k * math.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_type)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_type)
    G0 = G_p(q, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2


#--------------------------------------
# Eq 8 - GE
#--------------------------------------
def T_2(k, phi, mg, a1, a2, m2_type, q):
    q2 = q ** 2
    qk_cos = q * k * math.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_type)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_type)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)


#--------------------------------------
# Eq 11 - GE
#--------------------------------------
def born_sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

#--------------------------------------
# Eq 6 - GE
#--------------------------------------
def born_amp(diff_T, s, eps, t):
    alpha_pomeron = 1.0 + eps + 0.25 * t
    s_tilde = s/s0

    return 1j * s * 8 * (s_tilde**(alpha_pomeron - 1)) * diff_T

In [ ]:
def root_1d_integrator(func, lower_limit, upper_limit):
    """Integrates the given function using ROOT's adaptive integration method.
    args:
        func: The function to be integrated.
        lower_limit: Lower limit of integration.
        upper_limit: Upper limit of integration.
    returns:
        A tuple containing the estimated value of the integral and its uncertainty.
    """
    # creating Functor to be used by ROOT's integrator
    functor = ROOT.Math.Functor1D(func)

    type = ROOT.Math.IntegrationOneDim.kADAPTIVE   # integration type
    absTol = 1e-12
    relTol = 1e-12
    size   = 100000
    rule   = ROOT.Math.Integration.kGAUSS15   
        
    # defining Integrator object
    integrator = ROOT.Math.IntegratorOneDim(type, absTol, relTol, size, rule)
    integrator.SetFunction(functor)
    
    # calculating integral and error 
    result = integrator.Integral(lower_limit, upper_limit)
    error = integrator.Error()
    
    return result, error 



In [ ]:
def k_integral(k, mg, a1, a2, m2_func, q):
    """"Calculates the integral over k"""
    integrand = lambda phi: k * (T_1(k, phi, mg, a1, a2, m2_func, q) -
                                 T_2(k, phi, mg, a1, a2, m2_func, q))
    return integrand

def phi_integral(phi, mg, a1, a2, m2_func, q, k_max):
    """"Calculates the integral over phi"""
    def inner_in_k(k):
        return k * (T_1(k, phi, mg, a1, a2, m2_func, q) -
                    T_2(k, phi, mg, a1, a2, m2_func, q))
    
    result, _ = root_1d_integrator(inner_in_k, 0, k_max)
    return result

def compute_k_phi_integral(mg, a1, a2, m2_func, q, k_max):
    """Computes the double integral over k and phi."""
    result, error = root_1d_integrator(
        lambda phi: phi_integral(phi, mg, a1, a2, m2_func, q, k_max), 0, 2*math.pi)
    return result, error


# TESTING 

mg = param_mg_atlas_pl
a1 = param_a1_atlas_pl
a2 = param_a2_atlas_pl
q = 0
k_max = 13000

compute_k_phi_integral(mg, a1, a2, m2_pl, q, k_max)

In [ ]:
# COMPUTING SIGMA TOT BORN 

#start parameters
start_sqrt_s = 100
max_sqrt_s = 13000
step_size = 200

mg = param_mg_atlas_pl
a1 = param_a1_atlas_pl
a2 = param_a2_atlas_pl
q = 0

# start initial sqrt for while loop  
current_sqrt_s = start_sqrt_s

while current_sqrt_s <= max_sqrt_s:

    current_s = current_sqrt_s**2

    # calculates diff t (eq 7 and 8)
    diff_t,_ = compute_k_phi_integral(mg, a1, a2, m2_pl, q, current_sqrt_s)

    # calculating born amplitude
    born_amp_value = born_amp(diff_t, current_s, param_eps_atlas_pl, 0)

    #calculating sigma tot born
    born_sigma_tot_value = born_sigma_tot(born_amp_value, current_s)
    # print(born_sigma_tot_value)

    # increase step
    current_sqrt_s += step_size

In [ ]:
#--------------------------------------
# Eq 23 - EIK
#--------------------------------------

q_max = 0.2

def chi_eikonal(s, b, eps, mg, a1, a2, m2_func):
    """
    Eikonal function χ(s,b) from eq. (23)
    χ(s,b) = (1/s) ∫ q dq J₀(bq) A_Born(s,t)
    where t = -q²
    """
    def integrand_real(q):
        t = -q**2  # t = -q² as specified
        
        # Calculate diff_T for this q value        
        diff_T, _ = compute_k_phi_integral(
            mg=mg,
            a1=a1,
            a2=a2,
            m2_func=m2_func,
            q=q,
            k_max=np.sqrt(s)  # Use sqrt(s) as k_max
        )

        # Calculate Born amplitude for this t
        amp_born = born_amp(diff_T, s, eps, t)
        
        # Integrand: q * J₀(b*q) * A_Born(s,t)
        return q * j0(b * q) * amp_born.real
    
    def integrand_imag(q):
        t = -q**2  # t = -q² as specified
        
        # Calculate diff_T for this q value        
        diff_T, _ = compute_k_phi_integral(
            mg=mg,
            a1=a1,
            a2=a2,
            m2_func=m2_func,
            q=q,
            k_max=np.sqrt(s)  # Use sqrt(s) as k_max
        )

        # Calculate Born amplitude for this t
        amp_born = born_amp(diff_T, s, eps, t)
        
        # Integrand: q * J₀(b*q) * A_Born(s,t)
        return q * j0(b * q) * amp_born.imag
    
    real_integral_result, _ = root_1d_integrator(integrand_real, 0.0, q_max)  
    imag_integral_result, _ = root_1d_integrator(integrand_imag, 0.0, q_max)  

    integral_result = real_integral_result + 1j*imag_integral_result

    return integral_result / s

# print(chi_eikonal(7000**2, 100.0, param_eps_atlas_pl, param_mg_atlas_pl, param_a1_atlas_pl, param_a2_atlas_pl,m2_pl))

In [ ]:
# #--------------------------------------
# # Eq 24 - EIK
# #--------------------------------------

# b_max = 30  # Maximum impact parameter

# def eikonal_amplitude(s, t, eps, mg, a1, a2, m2_func):
#     """
#     Eikonalized amplitude from eq. (24)
#     A_eik(s,t) = i s ∫ b db J₀(b√(-t)) [1 - exp(iχ(s,b))]
#     where t is the Mandelstam variable (negative)
#     """
#     q = np.sqrt(-t)  # q = √(-t) since t = -q²
    
#     def integrand_real(b):
#         # Calculate eikonal function χ(s,b)
#         chi_val = chi_eikonal(s, b, eps, mg, a1, a2, m2_func)
        
#         # Compute [1 - exp(iχ(s,b))]
#         exp_term = np.exp(1j * chi_val)
#         one_minus_exp = 1.0 - exp_term
        
#         # Integrand: b * J₀(b√(-t)) * [1 - exp(iχ(s,b))]
#         return (b * j0(b * 0) * one_minus_exp).real
    
#     def integrand_imag(b):
#         # Calculate eikonal function χ(s,b)
#         chi_val = chi_eikonal(s, b, eps, mg, a1, a2, m2_func)
        
#         # Compute [1 - exp(iχ(s,b))]
#         exp_term = np.exp(1j * chi_val)
#         one_minus_exp = 1.0 - exp_term
        
#         # Integrand: b * J₀(b√(-t)) * [1 - exp(iχ(s,b))]
#         return (b * j0(b * 0) * one_minus_exp).imag
    
#     # Integrate real and imaginary parts separately
#     real_integral, _ = root_1d_integrator(integrand_real, 0.0, b_max)
#     imag_integral, _ = root_1d_integrator(integrand_imag, 0.0, b_max)
    
#     integral_result = real_integral + 1j * imag_integral
    
#     # Final amplitude: i s times the integral
#     return 1j * s * integral_result

# # print(eikonal_amplitude(7000**2,-0.04,param_eps_atlas_pl,param_mg_atlas_pl,param_a1_atlas_pl,param_a2_atlas_pl,m2_pl))

In [24]:
#--------------------------------------
# Eq 24 - EIK (with born_amp function as argument)
#--------------------------------------

b_max = 30  # Maximum impact parameter

def eikonal_amplitude_with_born_func(s, t, eps, mg, a1, a2, m2_func, born_amp_func, q_max=0.2):
    """
    Eikonalized amplitude from eq. (24)
    A_eik(s,t) = i s ∫ b db J₀(b√(-t)) [1 - exp(iχ(s,b))]
    where χ(s,b) is computed using the born_amp_func
    
    Parameters:
    - s: Mandelstam s (squared center-of-mass energy)
    - t: Mandelstam t (momentum transfer squared, negative)
    - eps, mg, a1, a2, m2_func: Model parameters
    - born_amp_func: Function that computes Born amplitude: born_amp(diff_T, s, eps, t)
    - q_max: Maximum q for χ integration
    """
    q = np.sqrt(-t)  # q = √(-t) since t = -q²
    
    def chi_eikonal_with_born_func(s, b, eps, mg, a1, a2, m2_func, born_amp_func, q_max):
        """
        Eikonal function χ(s,b) from eq. (23)
        χ(s,b) = (1/s) ∫ q dq J₀(bq) A_Born(s,t)
        where A_Born is computed using born_amp_func
        """
        def integrand_real(q_val):
            t_val = -q_val**2  # t = -q²
            
            # Calculate diff_T for this q value        
            diff_T, _ = compute_k_phi_integral(
                mg=mg,
                a1=a1,
                a2=a2,
                m2_func=m2_func,
                q=q_val,
                k_max=np.sqrt(s)
            )

            # Calculate Born amplitude using the provided function
            amp_born = born_amp_func(diff_T, s, eps, t_val)
            
            # Integrand: q * J₀(b*q) * A_Born(s,t)
            return (q_val * j0(b * q_val) * amp_born).real
    
        def integrand_imag(q_val):
            t_val = -q_val**2  # t = -q²
            
            # Calculate diff_T for this q value        
            diff_T, _ = compute_k_phi_integral(
                mg=mg,
                a1=a1,
                a2=a2,
                m2_func=m2_func,
                q=q_val,
                k_max=np.sqrt(s)
            )

            # Calculate Born amplitude using the provided function
            amp_born = born_amp_func(diff_T, s, eps, t_val)
            
            # Integrand: q * J₀(b*q) * A_Born(s,t)
            return (q_val * j0(b * q_val) * amp_born).imag
        
        real_integral_result, _ = root_1d_integrator(integrand_real, 0.0, q_max)  
        imag_integral_result, _ = root_1d_integrator(integrand_imag, 0.0, q_max)  

        integral_result = real_integral_result + 1j * imag_integral_result
        return integral_result / s
    
    def integrand_real(b_val):
        # Calculate eikonal function χ(s,b) using born_amp_func
        chi_val = chi_eikonal_with_born_func(s, b_val, eps, mg, a1, a2, m2_func, born_amp_func, q_max)
        
        # Compute [1 - exp(iχ(s,b))]
        exp_term = np.exp(1j * chi_val)
        one_minus_exp = 1.0 - exp_term
        
        # Integrand: b * J₀(b√(-t)) * [1 - exp(iχ(s,b))]
        return (b_val * j0(b_val * q) * one_minus_exp).real
    
    def integrand_imag(b_val):
        # Calculate eikonal function χ(s,b) using born_amp_func
        chi_val = chi_eikonal_with_born_func(s, b_val, eps, mg, a1, a2, m2_func, born_amp_func, q_max)
        
        # Compute [1 - exp(iχ(s,b))]
        exp_term = np.exp(1j * chi_val)
        one_minus_exp = 1.0 - exp_term
        
        # Integrand: b * J₀(b√(-t)) * [1 - exp(iχ(s,b))]
        return (b_val * j0(b_val * q) * one_minus_exp).imag
    
    # Integrate real and imaginary parts separately
    real_integral, _ = root_1d_integrator(integrand_real, 0.0, b_max)
    imag_integral, _ = root_1d_integrator(integrand_imag, 0.0, b_max)
    
    integral_result = real_integral + 1j * imag_integral
    
    # Final amplitude: i s times the integral
    return 1j * s * integral_result

print(eikonal_amplitude_with_born_func(7000**2,-0.04,param_eps_atlas_pl,param_mg_atlas_pl,param_a1_atlas_pl,param_a2_atlas_pl,m2_pl,eikonal_amplitude_with_born_func))


TypeError: Template method resolution failed:
  none of the 6 overloaded methods succeeded. Full details:
  double ROOT::Math::IntegratorOneDim::Integral(const ROOT::Math::IGenFunction& f, const vector<double>& pts) =>
    TypeError: could not convert argument 1
  double ROOT::Math::IntegratorOneDim::Integral(const vector<double>& pts) =>
    TypeError: takes at most 1 arguments (2 given)
  double ROOT::Math::IntegratorOneDim::Integral(const ROOT::Math::IGenFunction& f) =>
    TypeError: takes at most 1 arguments (2 given)
  double ROOT::Math::IntegratorOneDim::Integral() =>
    TypeError: takes at most 0 arguments (2 given)
  double ROOT::Math::IntegratorOneDim::Integral(const ROOT::Math::IGenFunction& f, double a, double b) =>
    TypeError: takes at least 3 arguments (2 given)
  double ROOT::Math::IntegratorOneDim::Integral(double a, double b) =>
    TypeError: Template method resolution failed:
  none of the 6 overloaded methods succeeded. Full details:
  double ROOT::Math::Integra